# Cauren bridge v4 hazard experiment

Rebuilds the v3 and v4 bridge datasets from pinned public bundles (FHWA NBI, USGS NSHM, FEMA NFHL; CC BY 4.0), runs the v3 tabular benchmark and the v4 discrete-time hazard models on the same test bridges, and copies the result files to `MyDrive/cauren_results`. Raw and derived data stay in the Colab VM.

Use **Runtime > Run all** and allow Google Drive access when asked.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content
!rm -rf cauren && git clone -q -b bridge-v4-dataset https://github.com/Ayberkrk/cauren.git
%cd /content/cauren
!git log --oneline -1
!pip -q install -e ".[dataset,benchmark]" pyarrow lightgbm

In [ ]:
!python tools/colab/run_bridge_v4.py --work /content/work 2>&1 | grep -v '^BRIDGE_V4_REPORT\|^{"v3_summary' | tail -n 60

In [ ]:
import shutil
import pathlib
out = pathlib.Path('/content/drive/MyDrive/cauren_results/bridge_v4')
out.mkdir(parents=True, exist_ok=True)
shutil.copy('/content/work/bridge_v4_report.json', out / 'bridge_v4_report.json')
for name in ('hazard_results.json', 'hazard_predictions.csv', 'calibration.json'):
    shutil.copy(f'/content/work/cauren_bridge_v4/hazard/{name}', out / name)
shutil.copy('/content/work/cauren_bridge/model_predictions.csv', out / 'v3_model_predictions.csv')
print(sorted(p.name for p in out.iterdir()))